In [ ]:
%load_ext autoreload
%autoreload 2

from collections import OrderedDict
from qiskit_metal import designs
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder

from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors

from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven #bonding
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround


from SQDMetal.Utilities.QUtilities import QUtilities
import matplotlib.pyplot as plt
import numpy as np
from qiskit_metal.toolbox_python.attr_dict import Dict

from toolkit.components.qubit.circle_transmon import CircTransmon

from scipy.constants import c, e, hbar
from scipy.constants import physical_constants as pc
# %matplotlib inline

In [11]:
res_len_lambda_2 = lambda eff, f: c/(2*np.sqrt(eff)*f)*1e3 # in mm
res_len_lambda_4 = lambda eff, f: c/(4*np.sqrt(eff)*f)*1e3 # in mm

In [2]:
design = designs.DesignPlanar({}, overwrite_enabled=True)

In [3]:
gui = MetalGUI(design)

In [4]:
options = Dict(
        # Position and orientation
        pos_x='1.8mm', pos_y='1.6mm', orientation=0, chip='main',
        cpw_width=design.variables['cpw_width'], cpw_gap=design.variables['cpw_gap'],

        # Pads (capacitor)
        pad_radius = '120um',
        pad_gap = '10um',

        # jj (mesoscopic: 0 or small junction: 1)
        jj_options=Dict(
            jj_width='5um',
            L_j = '40.7885nH',
            C_j = '0.72fF',
            export_mask = False,    # if True, draw a rectangle where the constriction will be made. If false, we keep a gap of jj_sim_gap in which we insert a jj element to make the simulation
            gds_cell_jj = 'gds_cell_jj',
            jj_sim_gap = '8um', # if we need a small junction, we can increase this gap
            ),
        
        # Coupled claws (2 claws allowed   )
        claw_a_options= Dict(
            make_claw=True,
            connector_location='90',  # 0 => 'west' arm, 90 => 'north' arm, 180 => 'east' arm
            claw_width='50um',
            claw_gap='30um',
            claw_length='200um',
            ground_spacing='10um',
            connector_length='70um',
            ),
        claw_b_options=Dict(
            make_claw=False,
            connector_location='0',  # 0 => 'west' arm, 90 => 'north' arm, 180 => 'east' arm
            claw_width='50um',
            claw_gap='30um',
            claw_length='100um',
            ground_spacing='10um',
            connector_length='50um',
            ),

        # Charge line
        charge_line=Dict(
            make_charge_line=True,
            location = '180', # 0 => 'west' arm, 90 => 'north' arm, 180 => 'east' arm. Must be different from claw_a and claw_b if they exist
            pad_radius='40um',
            pad_gap='40um', # gap between the pad and qubit
            ground_spacing='15um', # distance from the charge line to the ground
            line_length='100um',
            )
        )

# Create a fluxonium qubit with the Manucharyan design
qubit_1 = CircTransmon(design, 'Q1', options=options)


gui.rebuild()
gui.autoscale()

In [7]:
x1 = '0.25mm'
y1 = '3.2mm'

ops_1 = Dict(chip='main', pos_x=x1, pos_y=y1, orientation='360', lead_length='10um', 
             pad_width='120um', pad_gap='61um', trace_width=design.variables['cpw_width'] , trace_gap=design.variables['cpw_gap'] )
LP1 = LaunchpadWirebondDriven(design, 'LP1', options = ops_1)

# Driven Launchpad 2

x2 = '3mm'
y2 = '3.2mm'
ops_2 = Dict(chip='main', pos_x=x2, pos_y=y2, orientation='180', lead_length='10um',
             pad_width='120um', pad_gap='61um', trace_width=design.variables['cpw_width'] , trace_gap=design.variables['cpw_gap'] )
LP2 = LaunchpadWirebondDriven(design, 'LP2', options = ops_2)


# Rebuild the GUI
gui.rebuild()
gui.autoscale()

## Conecto LP1 con el extremo izquierdo del capacitor
opt_line = Dict(chip='main', trace_width =design.variables['cpw_width'] ,
        trace_gap =design.variables['cpw_gap'] , fillet='0um', hfss_wire_bonds = True, lead=Dict(end_straight='0.0mm'),
        pin_inputs=Dict(start_pin=Dict(component='LP1', pin='tie'),
                        end_pin=Dict(component='LP2', pin='tie')
                        )
               )

linea_1 = RoutePathfinder(design, 'linea_1', options = opt_line )

# Rebuild the GUI
gui.rebuild()
gui.autoscale() 


In [8]:
# charge lumped port
ops_charge_line = Dict(chip='main', pos_x='0.5mm', pos_y='1.6mm', orientation='0', lead_length='10um',
             pad_width='120um', pad_gap='61um', trace_width=design.variables['cpw_width'] , trace_gap=design.variables['cpw_gap'] )

LPC = LaunchpadWirebondDriven(design, 'LPC', options = ops_charge_line)

# Rebuild the GUI
gui.rebuild()
gui.autoscale()


In [9]:
# Connect LPC with charge_line pad
# anchors_points = [(0.7, 2), (0.7, 2.5), (0.8, 2.5), (1, 1.6)]
# anchors_dict = OrderedDict({i: point for i, point in enumerate(anchors_points)})

opt_line_charge = Dict(chip='main', trace_width =design.variables['cpw_width'],
        trace_gap = design.variables['cpw_gap'], hfss_wire_bonds = True, lead=Dict(end_straight='0.0mm'),
        pin_inputs=Dict(start_pin=Dict(component='LPC', pin='tie'),
                        end_pin=Dict(component='Q1', pin='charge_line')
                        )
                )

con_charge_line = RoutePathfinder(design, 'con_charge_line', options = opt_line_charge )
gui.rebuild()
gui.autoscale()

In [10]:
## Estimation for the resonator length
eff=(11.9+1)/2 ## Si and Air
f=7e9  # Resonator frequency in Hz

stg1 = ShortToGround(design, 'stg1', options=Dict(chip='main', pos_x='1mm',  pos_y='2.3mm', orientation='180'))

In [16]:
jogs = OrderedDict()
jogs[0] = ["L", '400um']
jogs[1] = ["L", '100um']
# jogs[1] = ["L", '300um']
# jogs[2] = ["R", '200um']
l = res_len_lambda_4(eff, f)  # in mm
total_length = f'{l}mm'

opt_meander = Dict(
    trace_width= design.variables['cpw_width'] ,
    trace_gap= design.variables['cpw_gap'] ,
    hfss_wire_bonds= True,
    meander=Dict(
        spacing= '250um',
        asymmetry= '0um'),    
    total_length= total_length,
    fillet = '90um',
    pin_inputs=Dict(
        end_pin=Dict(
            component= 'Q1',
            pin= 'claw_a'),
        start_pin=Dict(
            component= 'stg1',
            pin= 'short')),
    lead=Dict(
        start_straight='0.02mm',
        end_straight='1.278mm',
        end_jogged_extension=jogs
        ),
)
meandro_1 = RouteMeander(design, 'meandro_1',  options=opt_meander)

# rebuild the GUI
gui.rebuild()
gui.autoscale()

In [20]:
fig, axs = plt.subplots()
QUtilities.plot_highlight_component('Q1', design, ax=axs)

Simulations

In [ ]:

from SQDMetal.Comps.Labels import LabelText

design.delete_all_components()
############################
LabelText(design, 'leText', options=Dict(pos_x=0, pos_y=0,
                                    text='Clara put*', font_size='10um', font='serif'))
############################

fig, axs = plt.subplots(ncols=1); fig.set_figwidth(15)
QUtilities.plot_highlight_component('leText', design, ax=axs)

In [1]:
from SQDMetal.PALACE.Eigenmode_Simulation import PALACE_Eigenmode_Simulation
from SQDMetal.Utilities.Materials import MaterialInterface
import multiprocessing as mp

number_cpus = mp.cpu_count()

#Eigenmode Simulation Options
user_defined_options = {
                "mesh_refinement":  0,                             #refines mesh in PALACE - essetially divides every mesh element in half
                "dielectric_material": "silicon",                  #choose dielectric material - 'silicon' or 'sapphire'
                "starting_freq": 5e9,                              #starting frequency in Hz
                "number_of_freqs": 1,                              #number of eigenmodes to find
                "solns_to_save": 1,                                #number of electromagnetic field visualizations to save
                "solver_order": 2,                                 #increasing solver order increases accuracy of simulation, but significantly increases sim time
                "solver_tol": 1.0e-8,                              #error residual tolerance foriterative solver
                "solver_maxits": 200,                              #number of solver iterations
                "fillet_resolution":12,                            #number of vertices per quarter turn on a filleted path
                "palace_dir":"~/spack/opt/spack/linux-ubuntu24.04-zen2/gcc-13.3.0/palace-develop-36rxmgzatchgymg5tcbfz3qrmkf4jnmj/bin/palace",#"PATH/TO/PALACE/BINARY",
                "num_cpus": number_cpus                             #number of cpus used in simulation
                }

#Create the Palace Eigenmode simulation
eigen_sim = PALACE_Eigenmode_Simulation(name ='x-mon_test',                                         #name of simulation
                                        metal_design = design,                                      #feed in qiskit metal design
                                        sim_parent_directory = "",                                  #choose directory where mesh file and config file are saved
                                        mode = 'simPC',                                             #choose simulation mode 'HPC' or 'simPC'
                                        meshing = 'GMSH',                                           #choose meshing 'GMSH' or 'COMSOL'
                                        user_options = user_defined_options,                        #provide options chosen above
                                        create_files = True)                                        #create mesh, config and HPC batch files

#Add in metals from layer 1 of the design file
eigen_sim.add_metallic(1)

#Add in ground plane for simulation
eigen_sim.add_ground_plane()

#Add in the Josephson junction as a lumped port
eigen_sim.create_port_JosephsonJunction('junction', L_J=4.3e-9, C_J=10e-15)

#Fine-mesh x-mon
eigen_sim.fine_mesh_components(['x-mon'], min_size=8e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False)

#Sets up the lossy interfaces for MA, SA and MS interfaces
eigen_sim.setup_EPR_interfaces(metal_air=MaterialInterface('Aluminium-Vacuum'), substrate_air=MaterialInterface('Silicon-Vacuum'), substrate_metal=MaterialInterface('Silicon-Aluminium'))

#Prepares the mesh file and config file
eigen_sim.prepare_simulation()

NameError: name 'design' is not defined